# Data Consumption — Audio Classification (Audio Similarity Search)

Record 5 seconds of audio from the microphone, compute a **PANNs CNN14** embedding
(2048-dim), and search the Milvus `sound_audio_embeddings` collection for the
most acoustically similar recordings.

Prerequisites:
- Milvus running (`docker compose up -d milvus`)
- Audio embeddings ingested (orchestrator → option 6)

## Environment setup

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents]
     if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found :( ")

PROJECT_ROOT = str(_root)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)

## Constants

In [ ]:
import numpy as np

SAMPLE_RATE = 44100
DURATION = 5
TOP_K = 5

## Record audio — capture 5 s from the default microphone

In [ ]:
import sounddevice as sd

def record_audio(duration=DURATION, sample_rate=SAMPLE_RATE):
    """Record seconds of mono audio. Returns float32 array normalised to [-1, 1]."""
    print(f"\n  Recording {duration}s of audio (sample rate {sample_rate} Hz)...")
    print("  Speak / play a sound now!\n")

    audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate,
                   channels=1, dtype="float32")
    sd.wait()
    audio = audio.flatten()

    # Peak-normalise to [-1, 1] (same as warm-path).
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak

    rms = np.sqrt(np.mean(audio ** 2))
    print(f"  Recording complete — {len(audio)} samples, RMS={rms:.4f}")
    return audio

## Connect to Milvus

In [ ]:
from pymilvus import MilvusClient

MILVUS_URI = os.environ.get("MILVUS_URI", "http://localhost:19530")

milvus_client = MilvusClient(uri=MILVUS_URI)
print(f"Milvus connected: {MILVUS_URI}")
print(f"Collections: {milvus_client.list_collections()}")

## Compute PANNs CNN14 embedding (2048-dim)

In [ ]:
import torch
import torchaudio

def compute_audio_embedding(audio, sample_rate=SAMPLE_RATE):
    """Compute a 2048-dim PANNs CNN14 embedding from a float32 audio array."""
    # Resample to 32 kHz (PANNs expected input).
    waveform = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)
    if sample_rate != 32000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=32000)
        waveform = resampler(waveform)

    # Load PANNs CNN14 model.
    model = torch.hub.load("qiuqiangkong/panns_inference", "Cnn14", progress=False)
    model.eval()

    with torch.no_grad():
        output = model(waveform)
        embedding = output["embedding"].squeeze(0).numpy()

    # L2-normalise.
    norm = np.linalg.norm(embedding)
    if norm > 0:
        embedding = embedding / norm

    print(f"  Embedding shape: {embedding.shape}, L2 norm: {np.linalg.norm(embedding):.4f}")
    return embedding

## Search Milvus for similar audio

In [ ]:
AUDIO_COLLECTION = "sound_audio_embeddings"

def search_similar_audio(embedding, top_k=TOP_K):
    """Search the Milvus audio embedding collection for nearest neighbours."""
    results = milvus_client.search(
        collection_name=AUDIO_COLLECTION,
        data=[embedding.tolist()],
        limit=top_k,
        output_fields=["uuid", "category", "source", "peak_frequency_hz", "symmetry_score"],
        search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
    )
    return results[0] if results else []

## Display results

In [ ]:
def display_results(results):
    #Print search results.
    print(f"\n{'=' * 62}")
    print(f"  Audio Similarity Search Results — Top {len(results)}")
    print(f"{'─' * 62}")
    if not results:
        print("  No matching recordings found.")
        print(f"{'=' * 62}\n")
        return
    for i, hit in enumerate(results):
        entity = hit["entity"]
        print(f"  {i+1}. Similarity: {hit['distance']:.4f}")
        print(f"     UUID:       {entity.get('uuid', '?')}")
        print(f"     Category:   {entity.get('category', '') or '—'}")
        print(f"     Source:      {entity.get('source', '') or '—'}")
        print(f"     Peak freq:  {entity.get('peak_frequency_hz', 0):.0f} Hz")
        print(f"     Symmetry:   {entity.get('symmetry_score', 0):.3f}")
        print()
    print(f"{'=' * 62}\n")

## Run: Record → Embed → Search → Display

In [ ]:
# Step 1: Record audio from microphone.
audio = record_audio()

# Step 2: Compute PANNs CNN14 embedding.
embedding = compute_audio_embedding(audio)

# Step 3: Search Milvus for similar recordings.
results = search_similar_audio(embedding)

# Step 4: Display results.
display_results(results)